# Counterfactual Advantage Estimation

Given a branch state $s$ and two sampled continuations $a_A$ (original) and $a_B$ (counterfactual), we:

1. Show the reflector both hindsight trajectories from $s$ and extract forward-looking guidance $g$.
2. Score both actions under two policies:
   - $\pi_\text{base}(a \mid s)$ — the raw actor at the branch state.
   - $\pi_\text{ic}(a \mid s, g)$ — the same actor with $g$ injected into context.
3. Form signed advantage
   $$\Delta = \big[\log \pi_\text{ic}(a_A \mid s, g) - \log \pi_\text{base}(a_A \mid s)\big] - \big[\log \pi_\text{ic}(a_B \mid s, g) - \log \pi_\text{base}(a_B \mid s)\big].$$
4. $\text{sign}(\Delta)$ predicts which action the guided policy prefers. Compare to `preference` (`original` → A, `counterfactual` → B). Report sign-match rate on non-tie pairs.

Scoring uses the **full assistant message** (reasoning + action tag) at `branch_message_index`.

In [ ]:
import os, json, re, math, random
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Optional

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from transformers import AutoTokenizer
from tqdm.auto import tqdm

load_dotenv('../configs/vllm.env')

ACTOR_MODEL = os.environ['MODEL_STR']
ACTOR_PORT  = int(os.environ['VLLM_PORT'])
JUDGE_MODEL = os.environ['JUDGE_STR']
JUDGE_PORT  = int(os.environ['JUDGE_PORT'])
VLLM_KEY    = os.environ.get('VLLM_API_KEY', 'local-token')

actor_client = OpenAI(base_url=f'http://localhost:{ACTOR_PORT}/v1', api_key=VLLM_KEY)
judge_client = OpenAI(base_url=f'http://localhost:{JUDGE_PORT}/v1', api_key=VLLM_KEY)

print('actor :', ACTOR_MODEL, 'port', ACTOR_PORT)
print('judge :', JUDGE_MODEL, 'port', JUDGE_PORT)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(ACTOR_MODEL)
print('vocab size:', tokenizer.vocab_size)

## Load counterfactual pairs

In [ ]:
df = pd.read_json('counterfactual_pairs_100.jsonl', lines=True)
df_eval = df[df['preference'] != 'tie'].reset_index(drop=True)
print(f'total={len(df)}  non-tie={len(df_eval)}')
df_eval['preference'].value_counts()

## Hindsight extraction and reflector

Helpers from the design doc. The reflector sees both continuations in a randomized order and emits forward-looking guidance.

In [ ]:
TASK_HEADER = (
    'TASK CONTEXT:\n'
    'The agent is discovering a physical law P = f(k, A, delta_T, d) in a '
    'simulated universe where physics may differ from ours. Budget: 10 '
    'experiment rounds. Final submission is via <final_law>. Measurements '
    'are noise-free and deterministic. The agent may call <run_experiment>, '
    '<python>, or <final_law> (one action per turn).'
)

def _truncate_by_role(role, content, assistant_len=5000, other_len=4000):
    limit = assistant_len if role == 'assistant' else other_len
    if len(content) <= limit:
        return content
    return content[:limit] + f'\n... [truncated, {len(content) - limit} chars omitted]'

def _format_messages(messages, skip_system=True):
    lines = []
    for msg in messages:
        role = msg.get('role', 'unknown')
        if skip_system and role == 'system':
            continue
        content = _truncate_by_role(role, msg.get('content', ''))
        lines.append(f'[{role.upper()}]:\n{content}\n')
    return '\n'.join(lines)

def extract_action_block(content: str, agent_backend: str = '') -> str:
    """Best-effort action tag extraction (for display / logging only)."""
    for tag in ('final_law', 'run_experiment', 'python'):
        m = re.search(fr'<{tag}>.*?</{tag}>', content, flags=re.DOTALL)
        if m:
            return m.group(0)
    return content.strip()[:300]

def extract_hindsight_joint(pair: dict) -> dict:
    bi = pair['branch_message_index']
    backend = pair.get('agent_backend', 'vanilla_agent')
    ch_o = pair['original']['chat_history']
    ch_c = pair['counterfactual']['chat_history']
    return {
        'branch_turn': pair['branch_turn'],
        'branch_message_index': bi,
        'state_prefix': ch_o[:bi],
        'suffix_A': ch_o[bi:],
        'suffix_B': ch_c[bi:],
        'branch_msg_A': ch_o[bi],
        'branch_msg_B': ch_c[bi],
        'action_A': extract_action_block(ch_o[bi].get('content', ''), backend),
        'action_B': extract_action_block(ch_c[bi].get('content', ''), backend),
        'agent_backend': backend,
    }

def format_joint_prompt(h: dict, state_window: int = 4, randomize_order: bool = True, seed: Optional[int] = None):
    if seed is not None:
        random.seed(seed)
    branches = [('A', h['suffix_A']), ('B', h['suffix_B'])]
    if randomize_order:
        random.shuffle(branches)
    label_map = {f'Continuation {i+1}': lbl for i, (lbl, _) in enumerate(branches)}
    lines = [TASK_HEADER, '', '=' * 60, f"STATE AT DECISION POINT (turn {h['branch_turn']})", '=' * 60]
    lines.append(_format_messages(h['state_prefix'][-state_window:], skip_system=True))
    for i, (_, suffix) in enumerate(branches, start=1):
        lines.append('=' * 60)
        lines.append(f'CONTINUATION {i} FROM THIS STATE')
        lines.append('=' * 60)
        lines.append(_format_messages(suffix, skip_system=True))
    return '\n'.join(lines), label_map

SYSTEM_MSG_JOINT = (
    'You are a strategic advisor to a scientific-discovery agent. You will '
    'observe the state at a decision point and two possible continuations '
    'from that state. Your job is to extract transferable strategic guidance '
    'for an agent about to act at the state.\n\n'
    'Your guidance will be given to the agent BEFORE it acts. It must help '
    'the agent reason about which action to take — not endorse any specific '
    'action already observed in either continuation.\n\n'
    'Requirements:\n'
    '- Ground every recommendation in concrete features of the state: '
    'specific data values, parameter ranges, what has and has not been '
    'tested, what the current evidence does and does not support. Generic '
    "heuristics ('verify carefully', 'explore broadly') carry no signal and "
    'must be omitted.\n'
    "- Do not name or refer to 'Continuation 1' or 'Continuation 2'. Do not "
    "evaluate which continuation's agent did better. Phrase guidance as if "
    'the agent has not yet acted.\n'
    '- Identify what the continuations reveal about the task or state that '
    'a pre-action agent could not know. This is the actual content of '
    'hindsight — information about the state, not endorsement of actions.\n'
    "- If the state's evidence already determines a correct action (e.g., "
    'the law is identifiable from data already collected), say so directly '
    'and describe what a minimal completing action looks like.\n'
    '- Keep under 150 words. Write as forward-looking guidance, not '
    'retrospective narration.'
)

def call_joint_reflector(prompt_text: str, temperature: float = 0.6) -> str:
    user_msg = (
        f'{prompt_text}\n\n'
        'Produce strategic guidance for an agent about to act at the state '
        'above. Focus on what the continuations reveal about the task and '
        "state — not on which action either continuation's agent took."
    )
    resp = judge_client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {'role': 'system', 'content': SYSTEM_MSG_JOINT},
            {'role': 'user',   'content': user_msg},
        ],
        temperature=temperature,
        max_tokens=4000,
    )
    return resp.choices[0].message.content

## Scoring: log-probs of a branch assistant message under vLLM

We render the chat with the model's tokenizer, then hit vLLM's `/v1/completions` with `echo=True, logprobs=1` to get per-token logprobs of the prompt. Summing over the assistant-message token span gives $\log \pi(a \mid s)$.

- **Base prompt:** messages up to and including the branch assistant turn.
- **In-context prompt:** same, but with guidance $g$ inserted as a user turn immediately before the branch assistant turn.

In [ ]:
GUIDANCE_WRAPPER = (
    '[STRATEGIC ADVISOR NOTE — forward-looking guidance for your next action]\n'
    '{g}\n'
    '[END NOTE. Use this context as you decide your next action.]'
)

def build_prompts(state_prefix, branch_msg, guidance: Optional[str]):
    """Return (full_text, prefix_text). full_text ends with the assistant branch message.
    prefix_text is everything up to (not including) the branch message tokens, so that
    the token span [len(prefix_tokens), len(full_tokens)) covers exactly the assistant message.
    """
    msgs = list(state_prefix)
    if guidance is not None:
        msgs = msgs + [{'role': 'user', 'content': GUIDANCE_WRAPPER.format(g=guidance)}]
    prefix_text = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True,
    )
    full_msgs = msgs + [branch_msg]
    full_text = tokenizer.apply_chat_template(
        full_msgs, tokenize=False, add_generation_prompt=False,
    )
    assert full_text.startswith(prefix_text), 'prefix alignment failed — chat template inconsistency'
    return full_text, prefix_text

def score_action_logprob(state_prefix, branch_msg, guidance: Optional[str]) -> dict:
    """Sum token logprobs over the branch assistant message span under the actor."""
    full_text, prefix_text = build_prompts(state_prefix, branch_msg, guidance)
    prefix_ids = tokenizer(prefix_text, add_special_tokens=False)['input_ids']
    full_ids   = tokenizer(full_text,   add_special_tokens=False)['input_ids']
    action_len = len(full_ids) - len(prefix_ids)
    if action_len <= 0:
        return {'logprob': float('nan'), 'n_tokens': 0, 'note': 'empty action span'}
    resp = actor_client.completions.create(
        model=ACTOR_MODEL,
        prompt=full_text,
        max_tokens=1,
        temperature=0.0,
        echo=True,
        logprobs=1,
    )
    tok_logprobs = resp.choices[0].logprobs.token_logprobs  # len == len(full_ids); first entry is None
    # align: slice the last action_len logprobs corresponding to assistant tokens
    action_lps = [lp for lp in tok_logprobs[-action_len:] if lp is not None]
    return {
        'logprob': float(sum(action_lps)),
        'n_tokens': len(action_lps),
        'mean_logprob': float(sum(action_lps) / max(len(action_lps), 1)),
    }

## Per-pair advantage

In [ ]:
@dataclass
class PairResult:
    pair_index: int
    preference: str         # 'original' | 'counterfactual'
    gt_sign: int            # +1 if A (original) is better, -1 if B (counterfactual) is better
    guidance: str
    logp_base_A: float
    logp_base_B: float
    logp_ic_A: float
    logp_ic_B: float
    n_tok_A: int
    n_tok_B: int
    delta: float            # signed advantage, >0 prefers A
    pred_sign: int          # sign(delta)
    match: bool

def estimate_pair(pair: dict, reflector_seed: Optional[int] = 0) -> PairResult:
    h = extract_hindsight_joint(pair)
    prompt_text, _ = format_joint_prompt(h, randomize_order=True, seed=reflector_seed)
    guidance = call_joint_reflector(prompt_text)

    base_A = score_action_logprob(h['state_prefix'], h['branch_msg_A'], guidance=None)
    base_B = score_action_logprob(h['state_prefix'], h['branch_msg_B'], guidance=None)
    ic_A   = score_action_logprob(h['state_prefix'], h['branch_msg_A'], guidance=guidance)
    ic_B   = score_action_logprob(h['state_prefix'], h['branch_msg_B'], guidance=guidance)

    r_A = ic_A['logprob'] - base_A['logprob']
    r_B = ic_B['logprob'] - base_B['logprob']
    delta = r_A - r_B

    gt_sign = +1 if pair['preference'] == 'original' else -1
    pred_sign = 1 if delta > 0 else (-1 if delta < 0 else 0)

    return PairResult(
        pair_index=int(pair['pair_index']),
        preference=pair['preference'],
        gt_sign=gt_sign,
        guidance=guidance,
        logp_base_A=base_A['logprob'], logp_base_B=base_B['logprob'],
        logp_ic_A=ic_A['logprob'],     logp_ic_B=ic_B['logprob'],
        n_tok_A=base_A['n_tokens'],    n_tok_B=base_B['n_tokens'],
        delta=delta,
        pred_sign=pred_sign,
        match=(pred_sign == gt_sign),
    )

## Smoke test on one pair

In [ ]:
sample = df_eval.iloc[0].to_dict()
res = estimate_pair(sample)
print('preference :', res.preference, ' gt_sign:', res.gt_sign)
print('delta      :', res.delta)
print('pred_sign  :', res.pred_sign, ' match:', res.match)
print('\n--- guidance ---\n', res.guidance)

## Run across all non-tie pairs

In [ ]:
results = []
errors = []
for i, row in tqdm(list(df_eval.iterrows()), desc='pairs'):
    pair = row.to_dict()
    try:
        results.append(estimate_pair(pair, reflector_seed=pair['pair_index']))
    except Exception as e:
        errors.append((pair['pair_index'], repr(e)))
        print('error on pair', pair['pair_index'], e)

res_df = pd.DataFrame([asdict(r) for r in results])
out_path = Path('adv_est_results.jsonl')
res_df.to_json(out_path, orient='records', lines=True)
print(f'saved {len(res_df)} rows to {out_path}; errors={len(errors)}')

## Sign-match rate and breakdown

In [ ]:
import numpy as np
n = len(res_df)
acc = res_df['match'].mean()
print(f'overall sign-match: {acc:.3f}  (n={n})')
print('\nby ground-truth preference:')
print(res_df.groupby('preference')['match'].agg(['mean', 'count']))
print('\ndelta distribution:')
print(res_df['delta'].describe())
# separation: |delta| on matches vs misses
print('\n|delta| on matches vs misses:')
print(res_df.assign(abs_delta=res_df['delta'].abs()).groupby('match')['abs_delta'].describe())

In [ ]:
# quick bootstrap CI on sign-match
rng = np.random.default_rng(0)
B = 2000
boots = [rng.choice(res_df['match'].values, size=len(res_df), replace=True).mean() for _ in range(B)]
lo, hi = np.quantile(boots, [0.025, 0.975])
print(f'sign-match = {acc:.3f}  95% CI [{lo:.3f}, {hi:.3f}]')